In [3]:
# Cell 1: Configuración del entorno y rutas
import sys
import os
from pathlib import Path
import pandas as pd

# Definir la raíz del proyecto
project_root = Path(os.getcwd()).resolve().parent  # asume que estás en notebooks/
sys.path.append(str(project_root))

# Asegurar que src/ esté en sys.path
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))


In [4]:
# Cell 2: Importar módulos
from src.data_loading import cargar_datos_excel
from src.exploracion import estandarizar_columnas, convertir_fechas
from src.rentabilidad import calcular_rentabilidad, resumen_rentabilidad
from src.logistica import (
    calcular_tiempo_entrega,
    resumen_logistica,
    entregas_fuera_de_rango,
    eficiencia_entregas
)
from src.graficos_estadisticos import (
    grafico_barras_rentabilidad,
    grafico_linea_ventas,
    boxplot_margen
)
from src.exportacion import exportar_excel

from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas


In [ ]:
# Cell 3: Cargar y preparar los datos
ruta_excel = project_root / "data/raw/Sales report completa.xlsx"
df = cargar_datos_excel(str(ruta_excel), sheet_name="Ventas Supermercado")
df_productos = cargar_datos_excel(str(ruta_excel), sheet_name="Productos")
df_productos = estandarizar_columnas(df_productos)
df = estandarizar_columnas(df)
df = convertir_fechas(df)



   numero_venta  mes_salida  dia_salida  ano_salida  mes_entrega  dia_entrega  \
0          2040           1           1        2011            1            6   
1         47883           1           1        2011            1            8   
2          1220           1           1        2011            1            5   
3       3647632           1           1        2011            1            5   
4         47883           1           1        2011            1            8   

   ano_entrega          metodo_envio  numero_cliente   nombre_cliente  ...  \
0         2011        Standard Class           11280  Toby Braunhardt  ...   
1         2011  Standard      Class            15985      Joseph Holt  ...   
2         2011     Second      Class             735    Annie Thurman  ...   
3         2011          Second Class           14140     Eugene Moren  ...   
4         2011        Standard Class           15985      Joseph Holt  ...   

        pais       id_producto   ventas cant

In [6]:
# Cell 4: Análisis de rentabilidad
df = calcular_rentabilidad(df)
resumen_renta = resumen_rentabilidad(df)
resumen_renta.head()


,id_producto,total_venta,total_costo,utilidad,margen
0,FUR-ADV-10000002,159.12,98.73,60.39,0.3795
1,FUR-ADV-10000108,350.07,346.71,3.36,0.0096
2,FUR-ADV-10000183,974.83,1626.57,-651.74,-0.6506
3,FUR-ADV-10000188,124.95,120.75,4.20,-0.2195
4,FUR-ADV-10000190,222.36,117.90,104.46,0.4698


In [7]:
# Cell 5: Análisis logístico
df = calcular_tiempo_entrega(df)
resumen_log = resumen_logistica(df)
eficiencia = eficiencia_entregas(df)
df_fuera_rango = entregas_fuera_de_rango(df)
resumen_log.head()


,id_producto,pais,ciudad,costo_envio_sum,costo_envio_mean,tiempo_entrega_mean,tiempo_entrega_max,tiempo_entrega_min
0,FUR-ADV-10000002,Democratic Republic of the Congo,Kinshasa,6.11,6.11,5.0,5,5
1,FUR-ADV-10000002,Iraq,Baghdad,4.03,4.03,7.0,7,7
2,FUR-ADV-10000108,Liberia,Monrovia,9.27,9.27,2.0,2,2
3,FUR-ADV-10000108,Morocco,Casablanca,2.83,2.83,7.0,7,7
4,FUR-ADV-10000108,Rwanda,Kigali,10.80,10.80,4.0,4,4


In [8]:
# Cell 6: Exportar a Excel (opcional)
ruta_salida_excel = project_root / "data/processed/resumen_consolidado.xlsx"
exportar_excel(resumen_renta.merge(resumen_log, on="id_producto", how="outer"), ruta_salida_excel)


In [9]:
    # Cell 7: Exportar PDF con `reportlab`

import matplotlib.pyplot as plt

def exportar_pdf_resumen(nombre_pdf, resumen_renta, resumen_log, eficiencia):
        from reportlab.lib.pagesizes import letter
        from reportlab.pdfgen import canvas

        c = canvas.Canvas(nombre_pdf, pagesize=letter)
        width, height = letter
        y = height - 50

        c.setFont("Helvetica-Bold", 14)
        c.drawString(50, y, "Reporte Consolidado de Rentabilidad y Logística")
        y -= 30

        c.setFont("Helvetica", 10)
        c.drawString(50, y, f"Eficiencia en entregas: {eficiencia}%")
        y -= 20

        # --- Insertar gráfico de barras de rentabilidad ---
        from src.graficos_estadisticos import grafico_barras_rentabilidad
        grafico_path = "grafico_rentabilidad.png"
        grafico_barras_rentabilidad(resumen_renta, columna='utilidad', top_n=10)
        plt.savefig(grafico_path, bbox_inches='tight')
        plt.close()
        c.drawImage(grafico_path, 50, y-220, width=500, height=200)
        y -= 240

        c.setFont("Helvetica-Bold", 12)
        c.drawString(50, y, "Resumen Rentabilidad:")
        y -= 20

        for _, row in resumen_renta.head(10).iterrows():
            c.setFont("Helvetica", 10)
            texto = f"{row['id_producto']} - Venta: {row['total_venta']}, Utilidad: {row['utilidad']}, Margen: {row['margen']:.2%}"
            c.drawString(60, y, texto)
            y -= 15
            if y < 60:
                c.showPage()
                y = height - 50

        y -= 10
        c.setFont("Helvetica-Bold", 12)
        c.drawString(50, y, "Resumen Logístico:")
        y -= 20

        for _, row in resumen_log.head(10).iterrows():
            c.setFont("Helvetica", 10)
            texto = f"{row['id_producto']} - {row['pais']}/{row['ciudad']} - Costo Prom.: {row['costo_envio_mean']}, Tiempo Prom.: {row['tiempo_entrega_mean']:.1f} días"
            c.drawString(60, y, texto)
            y -= 15
            if y < 60:
                c.showPage()
                y = height - 50

        # --- Insertar gráfico de línea de ventas ---
        from src.graficos_estadisticos import grafico_linea_ventas
        grafico_path2 = "grafico_ventas.png"
        grafico_linea_ventas(df)
        plt.savefig(grafico_path2, bbox_inches='tight')
        plt.close()
        c.drawImage(grafico_path2, 50, 60, width=500, height=200)

        c.save()
        
        ruta_pdf = project_root / "data/reportes/reporte_consolidado.pdf"
        exportar_pdf_resumen(str(ruta_pdf), resumen_renta, resumen_log, eficiencia)
